# Rice variety classification with transfer learning

## 1. Environment and device setup

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf

physical_gpus = tf.config.list_physical_devices("GPU")
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs available: {len(physical_gpus)}")
print("Using GPU" if physical_gpus else "Using CPU")


## 2. Imports

In [ ]:
import gc
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import callbacks, layers

print("Imports completed.")


In [ ]:
tf.__version__

## 3. Load the dataset

In [ ]:
# The dataset path is discovered in the next section.


## 4. Inspect the dataset

In [ ]:
CLASSES = ["Arborio", "Basmati", "Ipsala", "Jasmine", "Karacadag"]

# Works with the attached Kaggle dataset and with a local copy.
candidates = [
    Path("/kaggle/input/datasets/muratkokludataset/rice-image-dataset/Rice_Image_Dataset"),
    Path("/kaggle/input/rice-image-dataset/Rice_Image_Dataset"),
    Path("data/Rice_Image_Dataset"),
]
BASE_DIR = next((path for path in candidates if path.exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Rice_Image_Dataset was not found in the expected locations.")

image_paths = []
labels = []
for rice_class in CLASSES:
    class_path = BASE_DIR / rice_class
    images = sorted(path for path in class_path.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(f"{rice_class}: {len(images)} images")
    image_paths.extend(str(path) for path in images)
    labels.extend([rice_class] * len(images))

df = pd.DataFrame({"filename": image_paths, "class": labels})
print(f"Total images: {len(df):,}")
print(df["class"].value_counts().sort_index())


## 5. Train, validation, and test split

In [ ]:
# ========================================
# STEP 4: DATA SPLIT (70/15/15)
# ========================================
print("Splitting data...")

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['class'],  # Maintain class balance
    random_state=42
)

# Second split: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['class'],
    random_state=42
)

print(f"✅ Train set:   {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"✅ Validation:  {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"✅ Test set:    {len(test_df)} images ({len(test_df)/len(df)*100:.1f}%)")

# Verify class balance in train set
print("\nClass distribution in train set:")
print(train_df['class'].value_counts().sort_index())

## 6. Image loading and augmentation

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64

preprocess_input = keras.applications.mobilenet_v2.preprocess_input

train_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

val_test_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="filename", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=True, seed=42
)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col="filename", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col="filename", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)

print(f"Class indices: {train_gen.class_indices}")


## 7. Transfer-learning model

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 8. Model training

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_accuracy", patience=7, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7, verbose=1
)

start_time = time.time()
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

minutes = (time.time() - start_time) / 60
print(f"Training time: {minutes:.1f} minutes")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")


## 9. Test-set evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

test_gen.reset()
y_true = test_gen.classes
y_pred = np.argmax(model.predict(test_gen, verbose=1), axis=1)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Test-set confusion matrix")
plt.tight_layout()
plt.show()


## 10. Save the model and run a prediction

In [ ]:
# ========================================
# STEP 9: SAVE MODEL & PREDICTION TOOL
# ========================================
# Save model
model.save('/kaggle/working/rice_classifier.h5')
print("✅ Model saved to: /kaggle/working/rice_classifier.h5")

# Prediction function
def predict_rice(img_path, model=model, classes=CLASSES, img_size=IMG_SIZE):
    """Predict rice type from image path"""
    from tensorflow.keras.preprocessing import image
    
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(img_size, img_size))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    preds = model.predict(img_array, verbose=0)[0]
    pred_idx = np.argmax(preds)
    confidence = preds[pred_idx]
    
    # Display result
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"{classes[pred_idx]} ({confidence*100:.1f}% confidence)", 
              fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    # Show all probabilities
    print("\nClass probabilities:")
    for i, cls in enumerate(classes):
        bar = '█' * int(preds[i] * 50)
        print(f"{cls:12s} | {bar} {preds[i]*100:5.2f}%")

# Example usage (uncomment to test with a sample image):
# sample_img = train_df['filename'].iloc[0]
# print(f"Testing with: {sample_img}")
# predict_rice(sample_img)

In [ ]:
# External image downloads are intentionally omitted from the public notebook.


In [ ]:
# External image downloads are intentionally omitted from the public notebook.


In [ ]:
# Example prediction using a held-out test image.
sample_path = test_df.iloc[0]["filename"]
predict_rice(sample_path)
